In [ ]:
#google drive의 COSE362-term-project/dataset 폴더와 연결
from google.colab import drive

drive.mount('/content/drive')

!ls /content/drive/MyDrive/COSE362-term-project/dataset

import sys

sys.path.append('/content/drive/MyDrive/COSE362-term-project/dataset')

import os

os.chdir("/content/drive/MyDrive/COSE362-term-project/dataset")

'COSE362-term-project (1)'
/content/drive/MyDrive/COSE362-term-project (1)/dataset
/content/drive/MyDrive/COSE362-term-project (1)/dataset


In [30]:
!ls

baseline  crop	landuse  Shapefiles


In [1]:
!pip install rasterstats


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


# **LIVESTOCK DATASET Preprocessing**

## dd

In [ ]:
from rasterstats import zonal_stats
import geopandas as gpd
import glob
import numpy as np

# 1. 지역구 지도(Shapefile)를 불러옵니다 (GROW-Africa)
regions = gpd.read_file("datasets/Shapefiles/GADM_level1_ECG.shp")

# 2. 1km 평균 tif 파일
# africa_1km_avg_{year}
livestock_afirca = "datasets/landuse/africa_1km_avg_2019.tif"

# 3. 구역 통계 실행 (sum/count 대신 'mean'을 요청)
#    (이 TIF 파일 자체가 0~256 사이의 평균값이므로, 'mean'을 구하면 됨)
print("Zonal statistics 시작... ")
stats = zonal_stats(regions, landuse_1km_raster, stats="mean")

# 4. 'mean' 값만 추출
mean_values = [s['mean'] if s and s['mean'] else 0 for s in stats]

# 5. 최종 '정착률' 계산 (0.0 ~ 1.0)
#    (픽셀 값이 0~256 사이의 평균값이므로, 256으로 나누면 비율이 됨)
regions['rate_landuse'] = [val / 256 for val in mean_values]

print("모든 계산 완료!")

sorted_df = regions.sort_values(by='rate_landuse',ascending=False)
print(sorted_df[['GID_1', 'COUNTRY','NAME_1','rate_landuse']].head())

In [3]:
import pandas as pd
import geopandas as gpd
from rasterstats import zonal_stats
import glob
import os
import time

print("--- 1. 기준 지도(Shapefile) 로드 ---")
# '마스터 키'가 될 Shapefile을 한 번만 불러옵니다.
shp_path = "dataset/livestock/Shapefiles/GADM_level1_ECG.shp"
regions = gpd.read_file(shp_path)

# L1_GID와 함께, 최종 테이블에 필요한 '이름' 컬럼들을 미리 준비합니다.
# (geometry는 무거우므로 제외하고, 중복을 제거합니다.)
# (GID_0은 country_code, NAME_1은 admin_1으로 가정합니다.)
lookup_df = regions[['GID_1', 'COUNTRY', 'GID_0', 'NAME_1']].drop_duplicates()
print(f"'{shp_path}'에서 총 {len(regions)}개의 지역(L1)과 {len(lookup_df)}개의 고유 ID를 로드했습니다.")


print("\n--- 2. 처리할 TIF 파일 검색 ---")
tif_folder = "dataset/livestock/africa_livestock/"
tif_files = glob.glob(f"{tif_folder}*.tif")

if not tif_files:
    print(f"🚨 [에러] '{tif_folder}' 폴더에서 .tif 파일을 찾지 못했습니다.")
else:
    print(f"총 {len(tif_files)}개의 TIF 파일을 찾았습니다.")
    
    all_results_list = []
    start_time = time.time()

    # --- 3. 모든 TIF 파일에 대해 Zonal Statistics 반복 실행 ---
    for i, tif_file in enumerate(tif_files, 1):
        
        filename = os.path.basename(tif_file)
        try:
            livestock_type, year_ext = filename.split('_')
            year = int(os.path.splitext(year_ext)[0])
        except ValueError:
            print(f"   [경고] '{filename}' 파일 이름 형식이 다릅니다. 건너뜁니다.")
            continue
            
        print(f"   ({i}/{len(tif_files)}) 처리 중: {livestock_type} / {year}년...")
        
        # Zonal Statistics 실행
        stats = zonal_stats(regions, tif_file, stats="mean")
        
        # GID_1과 결과(mean)를 매칭하여 리스트에 추가
        for j, s in enumerate(stats):
            mean_density = s['mean'] if s and s['mean'] is not None else 0
            gid = regions.iloc[j]['GID_1']
            
            all_results_list.append({
                'GID_1': gid,
                'Year': year,
                'Livestock': livestock_type,
                'Mean_Density': mean_density
            })

    print(f"\n--- 4. Zonal Statistics 완료 (총 소요 시간: {time.time() - start_time:.2f}초) ---")

    # --- 5. 피벗 테이블 생성 ---
    
    # 5a. "Long Format" DataFrame 생성
    df_long = pd.DataFrame(all_results_list)
    
    print("Pivoting: Long Format -> Wide Format으로 변환 중...")
    
    # 5b. Pivot 실행 (index: GID_1, Year)
    df_wide = df_long.pivot_table(
        index=['GID_1', 'Year'], # 먼저 GID와 Year로만 피벗합니다
        columns='Livestock',
        values='Mean_Density',
        fill_value=0
    ).reset_index()
    
    # --- 6. (핵심) 피벗 결과와 지역 이름(Lookup) 병합 ---
    print("Merging: 피벗 테이블과 지역(Shapefile) 정보 결합 중...")
    
    # GID_1을 키로 사용하여 df_wide와 lookup_df를 병합합니다.
    final_df = pd.merge(
        df_wide, 
        lookup_df, 
        on='GID_1', 
        how='left' # df_wide(데이터)를 기준으로, lookup(이름) 정보를 왼쪽에 붙임
    )

    # --- 7. 최종 컬럼 정리 및 저장 ---
    
    # 7a. 요청하신 순서대로 컬럼 목록 재정의
    # (가축 컬럼 이름은 피벗 과정에서 자동으로 생성됨)
    livestock_cols = [col for col in df_wide.columns if col not in ['GID_1', 'Year']]
    
    # 요청하신 컬럼 순서 + 가축 컬럼
    final_columns = [
        'COUNTRY',       # country
        'GID_0',         # country_code
        'NAME_1',        # admin_1
        'GID_1',         # L1_GID
        'Year'           # year
    ] + livestock_cols
    
    # 7b. 최종 DataFrame 생성 및 이름 변경
    final_df = final_df[final_columns].rename(columns={
        'COUNTRY': 'country',
        'GID_0': 'country_code',
        'NAME_1': 'admin_1',
        'GID_1': 'L1_GID',
        'Year': 'year'
    })
    
    # 7c. 최종 CSV 파일로 저장
    output_csv = "livestock_features_L1_detailed_index.csv"
    final_df.to_csv(output_csv, index=False)
    
    print(f"\n--- 6. ⭐️ 성공! 최종 피처 테이블 저장 완료 ---")
    print(f"파일 위치: {output_csv}")
    print("최종 테이블 미리보기 (상위 5줄):")
    print(final_df.head())
    

--- 1. 기준 지도(Shapefile) 로드 ---
'dataset/livestock/Shapefiles/GADM_level1_ECG.shp'에서 총 813개의 지역(L1)과 813개의 고유 ID를 로드했습니다.

--- 2. 처리할 TIF 파일 검색 ---
총 168개의 TIF 파일을 찾았습니다.
   (1/168) 처리 중: Buffa / 2018년...


/Users/sseongyunn/.pyenv/versions/3.12.2/lib/python3.12/site-packages/rasterstats/io.py:335: NodataWarning: Setting nodata to -999; specify nodata explicitly
  warnings.warn(


   (2/168) 처리 중: Horse / 2015년...
   (3/168) 처리 중: Horse / 2001년...
   (4/168) 처리 중: Horse / 2014년...
   (5/168) 처리 중: Buffa / 2019년...
   (6/168) 처리 중: Swine / 2019년...
   (7/168) 처리 중: Horse / 2002년...
   (8/168) 처리 중: Horse / 2016년...
   (9/168) 처리 중: Horse / 2017년...
   (10/168) 처리 중: Horse / 2003년...
   (11/168) 처리 중: Swine / 2018년...
   (12/168) 처리 중: Swine / 2020년...
   (13/168) 처리 중: Swine / 2008년...
   (14/168) 처리 중: Horse / 2007년...
   (15/168) 처리 중: Horse / 2013년...
   (16/168) 처리 중: Horse / 2012년...
   (17/168) 처리 중: Horse / 2006년...
   (18/168) 처리 중: Swine / 2009년...
   (19/168) 처리 중: Swine / 2021년...
   (20/168) 처리 중: Buffa / 2009년...
   (21/168) 처리 중: Buffa / 2021년...
   (22/168) 처리 중: Horse / 2010년...
   (23/168) 처리 중: Horse / 2004년...
   (24/168) 처리 중: Horse / 2005년...
   (25/168) 처리 중: Horse / 2011년...
   (26/168) 처리 중: Buffa / 2020년...
   (27/168) 처리 중: Buffa / 2008년...
   (28/168) 처리 중: Goats / 2010년...
   (29/168) 처리 중: Goats / 2004년...
   (30/168) 처리 중: Ducks / 20